1. Install Dependencies Cell

In [26]:
!pip install langgraph
!pip install langchain
!pip install langchain-google-genai
!pip install google-generativeai
!pip install fastapi uvicorn
!pip install PyGithub
!pip install python-dotenv
!pip install semgrep

2. Environment Variables Cell

In [3]:
import os

os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
os.environ["GITHUB_TOKEN"] = "YOUR_GITHUB_TOKEN"

3. Imports Cell

In [4]:
from typing import TypedDict, List, Dict

from langgraph.graph import StateGraph, END

from langchain_google_genai import ChatGoogleGenerativeAI

from github import Github

import json

4. Gemini LLM Setup Cell

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2
)

def ask_gemini(prompt):

    response = llm.invoke(prompt)

    return response.content

5. GitHub Client Setup Cell

In [6]:
github_client = Github(
    os.getenv("GITHUB_TOKEN")
)

C:\Users\harsh raj\AppData\Local\Temp\ipykernel_9108\3041667055.py:1: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  github_client = Github(


6. Define Global State Cell

In [7]:
class Finding(TypedDict):

    type: str
    severity: str
    confidence: float
    file: str
    line: int
    message: str
    suggestion: str


class PRState(TypedDict):

    repo: str
    pr_number: int

    changed_files: List[str]

    diff: str

    findings: List[Finding]

    ranked_findings: List[Finding]

    decision: str

    comments: List[Dict]

7. GitHub PR Fetcher Cell

In [8]:
def fetch_pr(repo_name, pr_number):

    repo = github_client.get_repo(repo_name)

    pr = repo.get_pull(pr_number)

    files = []

    patches = []

    for file in pr.get_files():

        files.append(file.filename)

        if file.patch:

            patches.append(file.patch)

    return {
        "files": files,
        "diff": "\n".join(patches)
    }

8. PR Intake Node Cell

In [9]:
def pr_intake_node(state):

    pr_data = fetch_pr(
        state["repo"],
        state["pr_number"]
    )

    state["changed_files"] = pr_data["files"]

    state["diff"] = pr_data["diff"]

    return state

9. Diff Intelligence Node Cell

In [10]:
def diff_intelligence_node(state):

    files = state["changed_files"]

    state["is_backend"] = any(
        f.endswith(".py")
        for f in files
    )

    state["is_frontend"] = any(
        f.endswith(".js") or f.endswith(".css")
        for f in files
    )

    state["has_auth_changes"] = any(
        "auth" in f.lower()
        for f in files
    )

    state["large_pr"] = len(files) > 10

    return state

10. JSON Parsing Helper Cell

In [11]:
def parse_json_response(response):

    try:

        response = response.strip()

        if response.startswith("```json"):

            response = response.replace("```json", "")
            response = response.replace("```", "")

        return json.loads(response)

    except Exception as e:

        print("JSON Parse Error:", e)

        return []

11. Bug Detector Agent Cell

In [12]:
def bug_agent(state):

    prompt = f"""
    Analyze this PR diff for:

    - logical bugs
    - null issues
    - race conditions
    - edge cases

    Return ONLY JSON ARRAY.

    Example:

    [
      {{
        "type": "logic_bug",
        "severity": "high",
        "confidence": 0.91,
        "file": "app.py",
        "line": 10,
        "message": "Possible null access",
        "suggestion": "Add null check"
      }}
    ]

    DIFF:
    {state['diff']}
    """

    response = ask_gemini(prompt)

    findings = parse_json_response(response)

    state["findings"].extend(findings)

    return state

12. Security Agent Cell

In [13]:
def security_agent(state):

    prompt = f"""
    Analyze this PR diff for:

    - SQL Injection
    - XSS
    - SSRF
    - auth bypass
    - secrets leakage
    - unsafe deserialization

    Return ONLY JSON ARRAY.

    DIFF:
    {state['diff']}
    """

    response = ask_gemini(prompt)

    findings = parse_json_response(response)

    state["findings"].extend(findings)

    return state

13. Performance Agent Cell

In [14]:
def perf_agent(state):

    prompt = f"""
    Detect performance problems:

    - N+1 queries
    - memory leaks
    - blocking IO
    - unnecessary loops
    - expensive rendering

    Return ONLY JSON ARRAY.

    DIFF:
    {state['diff']}
    """

    response = ask_gemini(prompt)

    findings = parse_json_response(response)

    state["findings"].extend(findings)

    return state

14. Code Smell Agent Cell

In [15]:
def smell_agent(state):

    prompt = f"""
    Analyze for:

    - readability
    - maintainability
    - SOLID violations
    - duplication
    - bad naming

    Return ONLY JSON ARRAY.

    DIFF:
    {state['diff']}
    """

    response = ask_gemini(prompt)

    findings = parse_json_response(response)

    state["findings"].extend(findings)

    return state

15. Ranking Node Cell

In [16]:
SEVERITY_SCORE = {
    "critical": 10,
    "high": 7,
    "medium": 5,
    "low": 2
}

def ranking_node(state):

    findings = state["findings"]

    ranked = sorted(
        findings,
        key=lambda x:
            SEVERITY_SCORE.get(
                x["severity"],
                1
            ) * x["confidence"],
        reverse=True
    )

    state["ranked_findings"] = ranked

    return state

16. Policy Decision Engine Cell

In [17]:
def policy_engine(state):

    findings = state["ranked_findings"]

    critical = len([
        f for f in findings
        if f["severity"] == "critical"
    ])

    high = len([
        f for f in findings
        if f["severity"] == "high"
    ])

    if critical > 0:

        state["decision"] = "REQUEST_CHANGES"

    elif high > 3:

        state["decision"] = "ESCALATE"

    elif len(findings) == 0:

        state["decision"] = "APPROVE"

    else:

        state["decision"] = "COMMENT_ONLY"

    return state

17. Comment Generator Cell

In [18]:
def comment_generator(state):

    comments = []

    for finding in state["ranked_findings"]:

        comments.append({
            "path": finding["file"],
            "line": finding["line"],
            "body": f"""
{finding['message']}

Suggestion:
{finding['suggestion']}
"""
        })

    state["comments"] = comments

    return state

18. GitHub Review Submitter Cell

In [19]:
def submit_review(state):

    repo = github_client.get_repo(
        state["repo"]
    )

    pr = repo.get_pull(
        state["pr_number"]
    )

    event_map = {
        "APPROVE": "APPROVE",
        "REQUEST_CHANGES": "REQUEST_CHANGES",
        "COMMENT_ONLY": "COMMENT"
    }

    review_event = event_map.get(
        state["decision"],
        "COMMENT"
    )

    pr.create_review(
        body=f"""
AI PR Review Result:
{state['decision']}
""",
        event=review_event,
        comments=state["comments"]
    )

    print("Review Submitted")

19. GitHub Action Node Cell

In [20]:
def github_action_node(state):

    submit_review(state)

    return state

20. Build LangGraph Workflow Cell

In [21]:
workflow = StateGraph(PRState)

workflow.add_node(
    "pr_intake",
    pr_intake_node
)

workflow.add_node(
    "diff_intelligence",
    diff_intelligence_node
)

workflow.add_node(
    "bug_agent",
    bug_agent
)

workflow.add_node(
    "security_agent",
    security_agent
)

workflow.add_node(
    "perf_agent",
    perf_agent
)

workflow.add_node(
    "smell_agent",
    smell_agent
)

workflow.add_node(
    "ranking",
    ranking_node
)

workflow.add_node(
    "policy",
    policy_engine
)

workflow.add_node(
    "comments",
    comment_generator
)

workflow.add_node(
    "github_review",
    github_action_node
)

workflow.set_entry_point(
    "pr_intake"
)

workflow.add_edge(
    "pr_intake",
    "diff_intelligence"
)

workflow.add_edge(
    "diff_intelligence",
    "bug_agent"
)

workflow.add_edge(
    "bug_agent",
    "security_agent"
)

workflow.add_edge(
    "security_agent",
    "perf_agent"
)

workflow.add_edge(
    "perf_agent",
    "smell_agent"
)

workflow.add_edge(
    "smell_agent",
    "ranking"
)

workflow.add_edge(
    "ranking",
    "policy"
)

workflow.add_edge(
    "policy",
    "comments"
)

workflow.add_edge(
    "comments",
    "github_review"
)

workflow.add_edge(
    "github_review",
    END
)

graph = workflow.compile()

21. Run Workflow Cell

In [22]:
state = {
    "repo": "YOUR_GITHUB_USERNAME/YOUR_REPO",
    "pr_number": 1,
    "findings": [],
    "comments": []
}

result = graph.invoke(state)

print(result["decision"])

BadCredentialsException: 401 {"message": "Bad credentials", "documentation_url": "https://docs.github.com/rest", "status": "401"}

22. View Findings Cell

In [23]:
for finding in result["ranked_findings"]:

    print("=" * 50)

    print("Type:", finding["type"])

    print("Severity:", finding["severity"])

    print("Confidence:", finding["confidence"])

    print("Message:", finding["message"])

    print("Suggestion:", finding["suggestion"])

NameError: name 'result' is not defined

23. Production Improvements Cell

In [24]:
"""
NEXT IMPROVEMENTS

1. Parallel execution
2. Redis caching
3. Semgrep integration
4. AST chunking
5. Human-in-loop
6. Embedding deduplication
7. LangSmith tracing
8. Async processing
9. Multi-LLM fallback
10. Auto-fix generation
"""

'\nNEXT IMPROVEMENTS\n\n1. Parallel execution\n2. Redis caching\n3. Semgrep integration\n4. AST chunking\n5. Human-in-loop\n6. Embedding deduplication\n7. LangSmith tracing\n8. Async processing\n9. Multi-LLM fallback\n10. Auto-fix generation\n'